Which models you tried, metrics comparison, threshold selection rationale

In [25]:
from pathlib import Path
import pandas as pd

In [26]:
df = pd.read_csv(Path.cwd().parent.parent / "archive" / "merged_data.csv")

In [27]:
df.head()

,id,date,client_id,card_id,amount,mcc,chip_transaction,online_transaction,swipe_transaction,bad_cvv,...,card_type,expires,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web,target,0
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,5499,0,0,1,0,...,Debit (Prepaid),2022-07,1,2,55.0,2008-05,2008,0,0.0,Miscellaneous Food Stores
1,7475328,2010-01-01 00:02:00,561,4575,14.57,5311,0,0,1,0,...,Credit,2024-12,1,1,9100.0,2005-09,2015,0,0.0,Department Stores
2,7475329,2010-01-01 00:02:00,1129,102,80.00,4829,0,0,1,0,...,Debit,2020-05,1,1,14802.0,2006-01,2008,0,0.0,Money Transfer
3,7475331,2010-01-01 00:05:00,430,2860,200.00,4829,0,0,1,0,...,Debit,2024-10,0,2,37634.0,2004-05,2006,0,NaN,Money Transfer
4,7475332,2010-01-01 00:06:00,848,3915,46.41,5813,0,0,1,0,...,Debit,2020-01,1,1,19113.0,2009-07,2014,0,0.0,Drinking Places (Alcoholic Beverages)


In [28]:
df.columns

Index(['id', 'date', 'client_id', 'card_id', 'amount', 'mcc',
       'chip_transaction', 'online_transaction', 'swipe_transaction',
       'bad_cvv', 'bad_cvv,insufficient_balance', 'bad_cvv,technical_glitch',
       'bad_card_number', 'bad_card_number,bad_cvv',
       'bad_card_number,bad_expiration',
       'bad_card_number,bad_expiration,insufficient_balance',
       'bad_card_number,insufficient_balance',
       'bad_card_number,technical_glitch', 'bad_expiration',
       'bad_expiration,bad_cvv', 'bad_expiration,insufficient_balance',
       'bad_expiration,technical_glitch', 'bad_pin',
       'bad_pin,insufficient_balance', 'bad_pin,technical_glitch',
       'bad_zipcode', 'bad_zipcode,insufficient_balance',
       'bad_zipcode,technical_glitch', 'insufficient_balance',
       'insufficient_balance,technical_glitch', 'none', 'technical_glitch',
       'current_age', 'yearly_income', 'credit_score', 'num_credit_cards',
       'is_female', 'is_retired', 'debt_to_income_ratio', 'car

In [29]:
df.drop(columns=['0', 'acct_open_date', 'year_pin_last_changed', 'client_id', 'card_id', 'amount', 'mcc', 'id', 'date', 'card_type', 'expires', 'card_brand'], inplace=True)

In [30]:
df.head()

,chip_transaction,online_transaction,swipe_transaction,bad_cvv,"bad_cvv,insufficient_balance","bad_cvv,technical_glitch",bad_card_number,"bad_card_number,bad_cvv","bad_card_number,bad_expiration","bad_card_number,bad_expiration,insufficient_balance",...,credit_score,num_credit_cards,is_female,is_retired,debt_to_income_ratio,has_chip,num_cards_issued,credit_limit,card_on_dark_web,target
0,0,0,1,0,0,0,0,0,0,0,...,740,4,1,0,2.281687,1,2,55.0,0,0.0
1,0,0,1,0,0,0,0,0,0,0,...,834,5,0,0,3.042873,1,1,9100.0,0,0.0
2,0,0,1,0,0,0,0,0,0,0,...,686,3,0,0,1.060698,1,1,14802.0,0,0.0
3,0,0,1,0,0,0,0,0,0,0,...,685,5,1,0,2.411921,0,2,37634.0,0,NaN
4,0,0,1,0,0,0,0,0,0,0,...,711,2,0,0,1.406951,1,1,19113.0,0,0.0


In [31]:
df = df.dropna(subset=["target"])

In [32]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, PrecisionRecallDisplay
import matplotlib.pyplot as plt

In [33]:
OHE_COLS = [c for c in df.columns if c not in [
    'current_age', 'yearly_income', 'credit_score', 'num_credit_cards',
    'is_female', 'is_retired', 'debt_to_income_ratio', 'has_chip',
    'num_cards_issued', 'credit_limit', 'card_on_dark_web', 'target'
]]

NUMERIC_COLS = ['current_age', 'yearly_income', 'credit_score',
                'num_credit_cards', 'debt_to_income_ratio',
                'num_cards_issued', 'credit_limit']
BINARY_COLS  = ['is_female', 'is_retired', 'has_chip', 'card_on_dark_web']
TARGET       = 'target'

In [34]:
print(f"OHE features:    {len(OHE_COLS)}")
print(f"Numeric:         {len(NUMERIC_COLS)}")
print(f"Binary flags:    {len(BINARY_COLS)}")
print(f"\nClass balance:\n{df[TARGET].value_counts(normalize=True).round(4)}")

OHE features:    26
Numeric:         7
Binary flags:    4

Class balance:
target
0.0    0.9985
1.0    0.0015
Name: proportion, dtype: float64


In [35]:
X = df[OHE_COLS + NUMERIC_COLS + BINARY_COLS]
y = df[TARGET]

In [36]:
# ── Diagnose first ──────────────────────────────────────────
print(f"Total rows       : {len(df)}")
print(f"NaN in target    : {df['target'].isna().sum()}")
print(f"NaN in features  : {X.isna().sum().sum()}")  # check X too while you're here

# Where are the NaN targets?
print(df[df['target'].isna()].head())

Total rows       : 8914963
NaN in target    : 0
NaN in features  : 0
Empty DataFrame
Columns: [chip_transaction, online_transaction, swipe_transaction, bad_cvv, bad_cvv,insufficient_balance, bad_cvv,technical_glitch, bad_card_number, bad_card_number,bad_cvv, bad_card_number,bad_expiration, bad_card_number,bad_expiration,insufficient_balance, bad_card_number,insufficient_balance, bad_card_number,technical_glitch, bad_expiration, bad_expiration,bad_cvv, bad_expiration,insufficient_balance, bad_expiration,technical_glitch, bad_pin, bad_pin,insufficient_balance, bad_pin,technical_glitch, bad_zipcode, bad_zipcode,insufficient_balance, bad_zipcode,technical_glitch, insufficient_balance, insufficient_balance,technical_glitch, none, technical_glitch, current_age, yearly_income, credit_score, num_credit_cards, is_female, is_retired, debt_to_income_ratio, has_chip, num_cards_issued, credit_limit, card_on_dark_web, target]
Index: []

[0 rows x 38 columns]


In [37]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # important for classification
)

In [38]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

In [39]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

binary_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_COLS),
        ("bin", binary_transformer, BINARY_COLS),
        ("cat", categorical_transformer, OHE_COLS)
    ]
)

In [40]:
logit_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

logit_model.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['current_age',
                                                   'yearly_income',
                                                   'credit_score',
                                                   'num_credit_cards',
                                                   'debt_to_income_ratio',
                                                   'num_cards_issued',
                                                   'credit_limit']),
                                                 ('bin',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_fre...
                                                   'bad_expiration,insufficient_balance',
                                                   'bad_expiration,technical_glitch',
                                                   'bad_pin',
                                                   'bad_pin,insufficient_balance',
                                                   'bad_pin,technical_glitch',
                                                   'bad_zipcode',
                                                   'bad_zipcode,insufficient_balance',
                                                   'bad_zipcode,technical_glitch',
                                                   'insufficient_balance',
                                                   'insufficient_balance,technical_glitch',
                                                   'none',
                                                   'technical_glitch'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [41]:
from lightgbm import LGBMClassifier

lgbm_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LGBMClassifier())
])

lgbm_model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 10666, number of negative: 7121304
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.240758 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1177
[LightGBM] [Info] Number of data points in the train set: 7131970, number of used features: 52
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001496 -> initscore=-6.503785
[LightGBM] [Info] Start training from score -6.503785


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['current_age',
                                                   'yearly_income',
                                                   'credit_score',
                                                   'num_credit_cards',
                                                   'debt_to_income_ratio',
                                                   'num_cards_issued',
                                                   'credit_limit']),
                                                 ('bin',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_fre...
                                                   'bad_expiration,bad_cvv',
                                                   'bad_expiration,insufficient_balance',
                                                   'bad_expiration,technical_glitch',
                                                   'bad_pin',
                                                   'bad_pin,insufficient_balance',
                                                   'bad_pin,technical_glitch',
                                                   'bad_zipcode',
                                                   'bad_zipcode,insufficient_balance',
                                                   'bad_zipcode,technical_glitch',
                                                   'insufficient_balance',
                                                   'insufficient_balance,technical_glitch',
                                                   'none',
                                                   'technical_glitch'])])),
                ('model', LGBMClassifier())])

In [42]:
y_pred_logit = logit_model.predict(X_test)
y_pred_lgbm = lgbm_model.predict(X_test)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [43]:
y_prob_logit = logit_model.predict_proba(X_test)[:, 1]
y_prob_lgbm = lgbm_model.predict_proba(X_test)[:, 1]

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [44]:
from sklearn.metrics import roc_auc_score

print("Logit AUC:", roc_auc_score(y_test, y_prob_logit))
print("LGBM AUC:", roc_auc_score(y_test, y_prob_lgbm))

Logit AUC: 0.8329315265869603
LGBM AUC: 0.8808293557917322


In [45]:
model = lgbm_model.named_steps["model"]
importances = model.feature_importances_

In [46]:
preprocessor = lgbm_model.named_steps["preprocess"]

In [47]:
feature_names = preprocessor.get_feature_names_out()

In [48]:
feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})

In [49]:
feature_importance_df = feature_importance_df.sort_values(
    by="importance",
    ascending=False
)

print(feature_importance_df.head(20))

## additions to features come from the pipeline namespaces, so num are numeric columns and cat are categorical columns

                        feature  importance
6             num__credit_limit         673
1            num__yearly_income         495
2             num__credit_score         375
0              num__current_age         354
4     num__debt_to_income_ratio         346
3         num__num_credit_cards         144
5         num__num_cards_issued          92
11      cat__chip_transaction_0          65
13    cat__online_transaction_0          64
9                 bin__has_chip          64
12      cat__chip_transaction_1          46
43               cat__bad_pin_0          43
7                bin__is_female          37
16     cat__swipe_transaction_1          29
15     cat__swipe_transaction_0          19
61      cat__technical_glitch_0          17
55  cat__insufficient_balance_0          15
44               cat__bad_pin_1          15
59                  cat__none_0          13
62      cat__technical_glitch_1          13


In [54]:

from sklearn.inspection import permutation_importance

result = permutation_importance(
    lgbm_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="roc_auc"
)

import pandas as pd

perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean
}).sort_values("importance", ascending=False)

print(perm_df.head(20))

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/

KeyboardInterrupt: 

In [55]:
from sklearn.metrics import roc_auc_score

baseline = roc_auc_score(y_test, lgbm_model.predict_proba(X_test)[:, 1])

drops = []

for col in X_test.columns:
    X_temp = X_test.copy()
    X_temp[col] = 0  # or median for numeric

    auc = roc_auc_score(y_test, lgbm_model.predict_proba(X_temp)[:, 1])

    drops.append((col, baseline - auc))

ablation_df = pd.DataFrame(drops, columns=["feature", "auc_drop"])
ablation_df.sort_values("auc_drop", ascending=False).head(20)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/

,feature,auc_drop
27,yearly_income,0.152739
32,credit_limit,0.100200
1,online_transaction,0.065273
28,credit_score,0.053669
26,current_age,0.033056
30,debt_to_income_ratio,0.029963
2,swipe_transaction,0.024001
0,chip_transaction,0.014259
29,num_credit_cards,0.012219
35,has_chip,0.008924
